In [4]:
from scipy.stats import randint, uniform, loguniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from import_folder import import_all_noiselevel_data
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer, precision_score
import pandas as pd

# -----------------------------------------------------------------------------
# Data Load and Definitions
# -----------------------------------------------------------------------------
folders = ['SNR_noiseless', 'SNR_50dB', 'SNR_40dB', 'SNR_30dB', 'SNR_20dB']

# Load all noise levels into a single DataFrame
all_data = import_all_noiselevel_data(folders, add_noise_column=False)

# Separate Features and Label
X = all_data.drop(columns=['label'])
y = all_data['label']
# Define cv strategy for hyperparameter tuning
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)



# random search parameters
num_combinations = 500  # Number of hyperparameter sets sampled

# other definitions
RANDOM_STATE = 42

# scoring function for normal precision
NORMAL_LABEL = "Normal" 
def normal_precision(y_true, y_pred):
    return precision_score(y_true, y_pred, labels=[NORMAL_LABEL], average="macro", zero_division=0)

normal_precision_scorer = make_scorer(normal_precision)

# Define scoring metric for hyperparameter tuning
scoring_metric = normal_precision_scorer

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [5]:
# -----------------------------------------------------------------------------
# Decision Tree (DT) Randomized Search
# -----------------------------------------------------------------------------
dt_param_dist = {
    "classifier__max_depth": randint(4, 25),
    "classifier__min_samples_split": randint(2, 20),
    "classifier__min_samples_leaf": randint(1, 10),
    "classifier__criterion": ["gini", "entropy"],
}

dt_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]
)

dt_random = RandomizedSearchCV(
    estimator=dt_pipeline,
    param_distributions=dt_param_dist,
    n_iter=num_combinations,  # Number of parameter settings sampled
    scoring={
            "normal_precision": normal_precision_scorer,
            "accuracy": "accuracy",
            "precision_macro": "precision_macro",
            "recall_macro": "recall_macro",
            "f1_macro": "f1_macro",
        },
    refit="normal_precision",
    cv=cv_strategy,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)


# -----------------------------------------------------------------------------
# Fit the Randomized Search on Training Data
# -----------------------------------------------------------------------------
print("Fitting Decision Tree Randomized Search...")
dt_random.fit(X_train, y_train)

print("\n=== TUNING COMPLETE ===")
print(f"Best DT CV {scoring_metric}: {dt_random.best_score_:.4f}")

# -----------------------------------------------------------------------------
# Inspect Best Hyperparameters Found by Randomized Search
# -----------------------------------------------------------------------------
print("=== BEST HYPERPARAMETERS FOUND ===")

print("\n1. Decision Tree (DT):")
for param, value in dt_random.best_params_.items():
    print(f"   {param}: {value}")


print("\n=== Ranking of the Best Hyperparameter Combinations ===")
# Get the results of the randomized search
dt_results = dt_random.cv_results_
random_results = pd.DataFrame(dt_results).sort_values(
    "mean_test_normal_precision", ascending=False
).reset_index(drop=True)

result_columns = [
    "params", "mean_test_normal_precision", "mean_test_accuracy",
    "mean_test_precision_macro", "mean_test_recall_macro", "mean_test_f1_macro",
]

display_results = random_results[result_columns]

Fitting Decision Tree Randomized Search...


/Users/felixvossbrink/Library/Mobile Documents/com~apple~CloudDocs/Uni/02 Master/01 Electrical Engineering/05 Quarter/01 Machine Learning/machine-learning/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/felixvossbrink/Library/Mobile Documents/com~apple~CloudDocs/Uni/02 Master/01 Electrical Engineering/05 Quarter/01 Machine Learning/machine-learning/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/felixvossbrink/Library/Mobile Documents/com~apple~CloudDocs/Uni


=== TUNING COMPLETE ===
Best DT CV make_scorer(normal_precision, response_method='predict'): 0.0000
=== BEST HYPERPARAMETERS FOUND ===

1. Decision Tree (DT):
   classifier__criterion: gini
   classifier__max_depth: 23
   classifier__min_samples_leaf: 8
   classifier__min_samples_split: 8

=== Ranking of the Best Hyperparameter Combinations ===
